# EDA

- [Introducción a la competición](#Titanic---Machine-Learning-from-Disaster)
- [Descripción del dataset](#Descripción-del-dataset)
- [Importar librerías](#Importar-librerías)
- [Lectura de datos](#Lectura-de-datos)
- [EDA](#Exploratory-Data-Analysis)
    - [Hipótesis previas](#Hipótesis-previas)
    - [Exploración](#Exporación-del-dataset)
    - [Análisis univariable](#Análisis-Univariable)
        - [Estadísticas descriptivas](#Estadísticas-descriptivas) 
        - [Visualizaciones](#Visualizaciones)
    - [Análisis multivariable](#Análisis-Multivariable)
    - [Testeo de hipótesis](#Testeo-Hipótesis)
- [Conclusiones](#Conclusiones)

## Titanic - Machine Learning from Disaster
El hundimiento del Titanic es uno de los naufragios más infames de la historia.

El 15 de abril de 1912, durante su viaje inaugural, el ampliamente considerado "insumergible" RMS Titanic se hundió tras chocar con un iceberg. Desafortunadamente, no había suficientes botes salvavidas para todos a bordo, lo que resultó en la muerte de 1502 de los 2224 pasajeros y tripulantes.

Si bien hubo un elemento de suerte en la supervivencia, parece que algunos grupos de personas tenían más probabilidades de sobrevivir que otros.

En este desafío, te pedimos que construyas un modelo predictivo que responda a la pregunta: "¿qué tipo de personas tenían más probabilidades de sobrevivir?" utilizando los datos de los pasajeros (es decir, nombre, edad, género, clase socioeconómica, etc.).

[Enlace a la competición en kaggle](https://www.kaggle.com/competitions/titanic)

## Descripción del dataset

Los datos se han dividido en dos grupos:

- Conjunto de entrenamiento (train.csv)
- Conjunto de prueba (test.csv)

El **conjunto de entrenamiento** debe utilizarse para construir tus modelos de machine learning. Para el conjunto de entrenamiento, proporcionamos el resultado (también conocido como "ground truth") para cada pasajero. Tu modelo se basará en "características" como el género y la clase de los pasajeros. También puedes usar ingeniería de características para crear nuevas variables.

El **conjunto de prueba** debe utilizarse para evaluar qué tan bien se desempeña tu modelo con datos no vistos. Para el conjunto de prueba, no proporcionamos la verdad real para cada pasajero. Es tu tarea predecir estos resultados. Para cada pasajero en el conjunto de prueba, utiliza el modelo que entrenaste para predecir si sobrevivió o no al hundimiento del Titanic.

### Diccionario de Datos

| **Variable** | **Definición**                                  | **Clave**                                      |
|:------------:|:-----------------------------------------------:|:----------------------------------------------:|
| survival     | Supervivencia                                   | `0 = No, 1 = Sí`                               |
| pclass       | Clase del boleto                                | `1 = 1ra, 2 = 2da, 3 = 3ra`                    |
| sex          | Sexo                                            |                                                |
| age          | Edad en años                                    |                                                |
| sibsp        | Número de hermanos/cónyuges a bordo del Titanic |                                                |
| parch        | Número de padres/hijos a bordo del Titanic      |                                                |
| ticket       | Número de boleto                                |                                                |
| fare         | Tarifa del pasajero                             |                                                |
| cabin        | Número de cabina                                |                                                |
| embarked     | Puerto de embarque                              | `C = Cherburgo, Q = Queenstown, S = Southampton` |


### Notas sobre las Variables

- **pclass**: Un indicador del estatus socioeconómico (SES)

    - 1ra = Upper

    - 2da = Middle

    - 3ra = Lower
      
- **age**: La edad es fraccionada si es menor a 1. Si la edad es estimada, aparece en la forma xx.5
  
- **sibsp**: El conjunto de datos define las relaciones familiares de esta manera...
  
    - Sibling = brother, sister, stepbrother, stepsister
  
    - Spouse = husband, wife (se ignoraron amantes y prometidos)
  
- **parch**: El conjunto de datos define las relaciones familiares de esta manera...
  
    - Parent = mother, father

    - Child = daughter, son, stepdaughter, stepson

Algunos niños viajaban solo con una niñera, por lo tanto, parch=0 para ellos.

## Importar librerías

In [ ]:
# Entorno de la asignatura: proyecto_ml (kernel "Python (proyecto_ml)").
# Ya incluye todas las librerías de este notebook: no hace falta instalar nada.

In [ ]:
import os
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency, ttest_ind

## Lectura de datos

In [ ]:
INPUT_ZIP = "./00_Data/Raw/titanic.zip"  # Directorio del zip
OUTPUT_FOLDER = "./00_Data/Raw"  # Directorio de destino
TRAIN_FILENAME = "train.csv"  # Nombre del fichero de entrenamiento

def fetch_data(input_path=INPUT_ZIP, output_dir=OUTPUT_FOLDER):
    """
    Extrae el contenido de un archivo ZIP en un directorio de destino.

    Parámetros:
    -----------
    input_path : str, opcional
        Ruta al archivo ZIP que se desea descomprimir. El valor predeterminado es la variable 'INPUT_ZIP'.
        
    output_dir : str, opcional
        Directorio en el cual se extraerá el contenido del archivo ZIP. Si el directorio no existe,
        será creado automáticamente. El valor predeterminado es la variable 'OUTPUT_FOLDER'.

    Comportamiento:
    ---------------
    - Crea el directorio de destino si no existe.
    - Descomprime el archivo ZIP en el directorio de destino.

    Excepciones:
    ------------
    Puede lanzar una excepción si el archivo ZIP no existe o si hay problemas al descomprimirlo.

    Ejemplo de uso:
    ---------------
    fetch_data('data.zip', 'output/')
    """
    # Comprobación de que el directorio de destino existe
    os.makedirs(output_dir, exist_ok=True)

    # Descomprime el archivo ZIP en caso de que no haya ningún csv en la carpeta
    if(len([file for file in os.listdir(output_dir) if file.endswith('.csv')]) == 0):
        with zipfile.ZipFile(input_path, 'r') as zip_ref:
            zip_ref.extractall(output_dir)


def load_data(directory=OUTPUT_FOLDER, filename=TRAIN_FILENAME):
    """
    Lee un archivo CSV desde el directorio especificado.

    Parámetros:
    -----------
    directory : str
        El directorio donde se encuentra el archivo CSV.
        
    filename : str
        El nombre del archivo CSV a leer (incluyendo la extensión .csv).

    Retorna:
    --------
    pd.DataFrame
        Un DataFrame de pandas que contiene los datos del archivo CSV.

    Excepciones:
    ------------
    FileNotFoundError:
        Se lanza si el archivo no existe en el directorio dado.
    
    Ejemplo de uso:
    ---------------
    df = read_csv_from_directory('data', 'file.csv')
    """
    # Construir la ruta completa al archivo CSV
    file_path = os.path.join(directory, filename)

    # Verificar si el archivo existe
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"El archivo {filename} no se encuentra en el directorio {directory}")

    # Leer el archivo CSV en un DataFrame
    return pd.read_csv(file_path)

fetch_data()
df = load_data()
df

## Hipótesis previas

1. **Hipótesis de Género**: Las mujeres tuvieron una mayor probabilidad de sobrevivir que los hombres debido al protocolo de evacuación de "mujeres y niños primero".

2. **Hipótesis de Clase**: Los pasajeros de clases más altas (`Pclass = 1`) tuvieron una mayor probabilidad de sobrevivir en comparación con los pasajeros de clases más bajas, reflejando las desigualdades sociales de la época.

3. **Hipótesis de Edad**: Los pasajeros más jóvenes (niños) tuvieron una mayor probabilidad de sobrevivir debido a que se les dio prioridad en los esfuerzos de rescate.

4. **Hipótesis de Tamaño de la Familia**: Los pasajeros que viajaban con familia tuvieron mayores probabilidades de sobrevivir en comparación con aquellos que viajaban solos. Sin embargo, familias muy grandes pudieron tener dificultades para escapar juntas, reduciendo sus posibilidades de supervivencia.

5. **Hipótesis de la Tarifa**: Los pasajeros que pagaron tarifas más altas tuvieron una mayor probabilidad de sobrevivir, ya que es probable que estuvieran en camarotes de clases superiores y tuvieran mejor acceso a los botes salvavidas.

6. **Hipótesis del Punto de Embarque**: Los pasajeros que embarcaron en ciertos puntos, como Cherburgo (`Embarked = C`), tuvieron mayores probabilidades de sobrevivir debido a diferencias en riqueza o ubicación de los camarotes.

7. **Hipótesis de Disponibilidad de Cabina**: Los pasajeros con información de cabina disponible (`Cabin` no nulo) tuvieron una mayor probabilidad de sobrevivir, ya que probablemente estaban en las secciones de clase alta del barco.

8. **Hipótesis de Título**: Los pasajeros con ciertos títulos, como `Mrs`, `Miss` o `Master` (niños), tuvieron mejores probabilidades de sobrevivir que otros, reflejando su estatus social o edad.

9. **Hipótesis del Número de Ticket**: Los pasajeros con números de ticket similares, lo que podría indicar que viajaban juntos, tuvieron probabilidades de supervivencia correlacionadas.

10. **Hipótesis de Interacciones**:
    - **Género y Clase**: Las mujeres de clases más altas tuvieron la mayor tasa de supervivencia.
    - **Edad y Clase**: Los niños de clases más bajas tuvieron menos probabilidades de sobrevivir que los niños de clases más altas.
    - **Tamaño de Familia y Tarifa**: Los pasajeros con familias más pequeñas y tarifas más altas tuvieron mayores probabilidades de supervivencia.

## Exploratory Data Analysis

### Exporación del dataset

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

#### Comentarios:
- Existen missing values en 'Age', 'Cabin' y 'Embarked' con los que tendremos que tratar.
- Los tipos de las variables parecen correctos, aunque 'Age' debería ser entero y aparece como float --> posiblemente por los NaNs
- La columna 'Name' no aporta mucho por sí misma, veremos si podemos extraer algo de ésta que sí valga.

### Análisis Univariable

#### Estadísticas descriptivas

In [ ]:
df.describe().T

In [ ]:
# Calcular moda
df.mode().T

In [ ]:
# Calcular moda (elimino 'Name' & 'PassengerID' para eliminar múltiples columnas)
df.drop(columns=["Name", "PassengerId"]).mode().T

In [ ]:
# Calcular IQR
df.describe().T["75%"] - df.describe().T["25%"]

In [ ]:
# Calcular rango
df.describe().T["max"] - df.describe().T["min"]

In [ ]:
# Almacenar columnas numéricas para evitar error
num_columns = df.describe().columns
num_columns

In [ ]:
# Calcular asimetría
"""
valor > 0 -> distribución asimétrica a la derecha
valor = 0 -> distribución simétrica
valor < 0 -> distribución asimétrica a la izquierda
"""
df[num_columns].skew()

In [ ]:
# Calcular la curtosis
"""
valor > 0 -> distribución leptocúrtica
valor = 0 -> distribución mesocúrtica
valor < 0 -> distribución platicúrtica
"""
df[num_columns].kurt()

#### Visualizaciones

In [ ]:
# Pintamos solo las variables con sentido (excluimos identificadores y texto libre)
cols_plot = [c for c in df.columns if c not in ['PassengerId', 'Name', 'Ticket', 'Cabin']]
for column in cols_plot:
    plt.figure(figsize=(8, 4))
    if pd.api.types.is_numeric_dtype(df[column]):
        sns.histplot(df[column].dropna(), bins=30)
        plt.title(f'Histograma de {column}')
        plt.ylabel('Frecuencia')
    else:
        sns.countplot(x=df[column])
        plt.title(f'Conteo de {column}')
        plt.ylabel('Total')
    plt.tight_layout()
    plt.show()

In [ ]:
# Genera boxplots para cada columna numérica del DataFrame
df.boxplot(figsize=(10, 8), rot=45)
plt.title('Boxplots de todas las columnas')
plt.tight_layout()  # Para ajustar mejor el espacio de la gráfica
plt.show()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Seleccionamos solo las columnas numéricas
numeric_columns = df.select_dtypes(include=['float', 'int']).columns
df_numeric = df[numeric_columns]

# Normalizamos los datos entre 0 y 1
scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_numeric), columns=numeric_columns)

# Genera boxplots para las columnas escaladas
df_scaled.boxplot(figsize=(10, 8), rot=45)
plt.title('Boxplots de columnas escaladas')
plt.tight_layout()
plt.show()

#### Comentarios:
Los valores parecen ser coherentes.

### Análisis Multivariable

In [ ]:
sns.pairplot(df);

In [ ]:
# Correlaciones
df[num_columns].corr()

In [ ]:
# Defino un cmap para ver más claramente los colores
cmap = sns.diverging_palette(500, 10, as_cmap=True)

In [ ]:
plt.figure(figsize=(10,8), dpi=110)
sns.heatmap(df[num_columns].corr(method='pearson'),
            annot=True, 
            fmt=".2f", linewidth=.5, cmap=cmap, vmin=-1, vmax=1)
plt.title("Correlación de Pearson", fontsize =20)
plt.show()

In [ ]:
plt.figure(figsize=(10,8), dpi=110)
sns.heatmap(df[num_columns].corr(method='spearman'),
            annot=True, 
            fmt=".2f", linewidth=.5, cmap=cmap, vmin=-1, vmax=1)
plt.title("Correlación de Spearman", fontsize =20)
plt.show()

In [ ]:
plt.figure(figsize=(10,8), dpi=110)
sns.heatmap(df[num_columns].corr(method='kendall'),
            annot=True, 
            fmt=".2f", linewidth=.5, cmap=cmap, vmin=-1, vmax=1)
plt.title("Correlación de Kendall", fontsize =20)
plt.show()

#### Comentarios:
No parece que haya correlaciones demasiado fuertes entre variables, salvo la trivial (clase-tarifa)

### Testeo Hipótesis

> **Cómo testear cada hipótesis (guía):** para cada una, apóyate en un **gráfico** (p. ej. `pd.crosstab(..., normalize='index').plot(kind='bar', stacked=True)` o un `groupby('...')['Survived'].mean()`) y confírmalo con una **prueba estadística**:
> - Variables **categóricas** (Sex, Pclass, Embarked, Has_Cabin, Title…): **chi²** con `scipy.stats.chi2_contingency` sobre `pd.crosstab(var, Survived)`.
> - Variables **numéricas** (Age, Fare…): **t-test** con `scipy.stats.ttest_ind` comparando supervivientes vs no supervivientes.
>
> Interpreta el p-valor: si **p < 0.05**, la relación/diferencia es estadísticamente significativa.

1. **Hipótesis de Género**: Las mujeres tuvieron una mayor probabilidad de sobrevivir que los hombres debido al protocolo de evacuación de "mujeres y niños primero".

In [ ]:
# @TODO: testear hipótesis

2. **Hipótesis de Clase**: Los pasajeros de clases más altas (`Pclass = 1`) tuvieron una mayor probabilidad de sobrevivir en comparación con los pasajeros de clases más bajas, reflejando las desigualdades sociales de la época.

In [ ]:
# @TODO: testear hipótesis

3. **Hipótesis de Edad**: Los pasajeros más jóvenes (niños) tuvieron una mayor probabilidad de sobrevivir debido a que se les dio prioridad en los esfuerzos de rescate.

In [ ]:
# @TODO: testear hipótesis

4. **Hipótesis de Tamaño de la Familia**: Los pasajeros que viajaban con familia tuvieron mayores probabilidades de sobrevivir en comparación con aquellos que viajaban solos. Sin embargo, familias muy grandes pudieron tener dificultades para escapar juntas, reduciendo sus posibilidades de supervivencia.

In [ ]:
# @TODO: testear hipótesis

5. **Hipótesis de la Tarifa**: Los pasajeros que pagaron tarifas más altas tuvieron una mayor probabilidad de sobrevivir, ya que es probable que estuvieran en camarotes de clases superiores y tuvieran mejor acceso a los botes salvavidas.

In [ ]:
# @TODO: testear hipótesis

6. **Hipótesis del Punto de Embarque**: Los pasajeros que embarcaron en ciertos puntos, como Cherburgo (`Embarked = C`), tuvieron mayores probabilidades de sobrevivir debido a diferencias en riqueza o ubicación de los camarotes.

In [ ]:
# @TODO: testear hipótesis

7. **Hipótesis de Disponibilidad de Cabina**: Los pasajeros con información de cabina disponible (`Cabin` no nulo) tuvieron una mayor probabilidad de sobrevivir, ya que probablemente estaban en las secciones de clase alta del barco.

In [ ]:
# @TODO: testear hipótesis

8. **Hipótesis de Título**: Los pasajeros con ciertos títulos, como `Mrs`, `Miss` o `Master` (niños), tuvieron mejores probabilidades de sobrevivir que otros, reflejando su estatus social o edad.

In [ ]:
# @TODO: testear hipótesis

9. **Hipótesis del Número de Ticket**: Los pasajeros con números de ticket similares, lo que podría indicar que viajaban juntos, tuvieron probabilidades de supervivencia correlacionadas.

In [ ]:
# @TODO: testear hipótesis

10. **Hipótesis de Interacciones**:
    - **Género y Clase**: Las mujeres de clases más altas tuvieron la mayor tasa de supervivencia.
    - **Edad y Clase**: Los niños de clases más bajas tuvieron menos probabilidades de sobrevivir que los niños de clases más altas.
    - **Tamaño de Familia y Tarifa**: Los pasajeros con familias más pequeñas y tarifas más altas tuvieron mayores probabilidades de supervivencia.

In [ ]:
# @TODO: testear hipótesis

## Conclusiones

In [ ]:
# @TODO: recapitular conclusiones